# Update 4 experiments: feature 7 (port window), the SYN swap, and Random Forest

COMP6310 group project. This notebook continues `ddos_xgboost_starter_v2.ipynb`, which is unchanged.
Sections 0 to 6 are copied from v2 so the data pipeline is the same; the only additions are reading
`Dst Port` and building feature 7.

On 31 Oct 2026 the group settled on six feature groups. This notebook:

- builds **feature 7** in a port version: `dst_port_flows_10s` = the number of flows of the same file to
  the same destination port that started in the 10 whole seconds before the flow. The group's feature 7
  used the destination IP, which the 21 Feb file does not have;
- tests **replacing SYN flag count** (feature 5) with feature 7, or with one of features 8 to 15, using the
  group's rule from Week 3: a feature helps only if recall improves by more than the spread between
  seeds and false alarms do not rise;
- compares **XGBoost with Random Forest and logistic regression** on the same splits.

Settings are those of the desktop app's training script (`DDoSDetector/train/train_model.py`): 300 trees,
depth 6, learning rate 0.1, benign flows capped at 1,000,000 in training, threshold 0.5, seeds 0 and 1.
The model score is the model's raw output, not calibrated, so it is not a probability.

Running the whole notebook takes about an hour (reading the two files takes a couple of minutes; the
rest is model training).

## 0. Imports

In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

## 1. Settings (as in v2, plus the destination port for feature 7)

In [2]:
DATA_DIR = Path("../data")      # put the CSV files here; do not commit them
FILES = ["Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv",
         "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv"]

LABEL, TIME = "Label", "Timestamp"
PORT = "Dst Port"               # feature 7 only: used to count flows, never a model input
WINDOW_SECONDS = 10
BASE_FEATURES = [
    "Flow Pkts/s", "Flow Byts/s", "Flow Duration",
    "Flow IAT Mean", "Flow IAT Std", "SYN Flag Cnt",
    "Fwd Pkt Len Mean", "Fwd Pkt Len Std",
]
COUNT_COLS = ["Tot Fwd Pkts", "Tot Bwd Pkts"]   # used to build the fwd/bwd ratio feature
CANDIDATE_COLS = ["TotLen Fwd Pkts", "TotLen Bwd Pkts", "Fwd IAT Mean", "Fwd IAT Min",
                  "Active Mean", "Idle Mean", "PSH Flag Cnt", "Fwd Act Data Pkts"]
EXTRA_COLS = ["Bwd Pkt Len Mean", "RST Flag Cnt", "FIN Flag Cnt"] + CANDIDATE_COLS

SEEDS = [0, 1]
BENIGN_CAP = 1_000_000          # benign flows in each training set (as train_model.py)
TRAIN_FRAC = 0.7

## 2. Load the data (v2's loader; `Dst Port` is kept as text)

In [3]:
def load(path):
    head = pd.read_csv(path, nrows=0).columns.tolist()
    print(path.name, "-", len(head), "columns | has Dst Port:", PORT in head)
    for need in [LABEL, TIME, PORT] + BASE_FEATURES + COUNT_COLS:
        if need not in head:
            print("   MISSING column:", need)
    for opt in EXTRA_COLS:
        if opt not in head:
            print("   optional column missing (skipped):", opt)
    wanted = [LABEL, TIME, PORT] + BASE_FEATURES + COUNT_COLS + EXTRA_COLS
    keep = [c for c in head if c in set(wanted)]
    df = pd.read_csv(path, usecols=keep, low_memory=False, dtype={PORT: str})
    df = df[df[LABEL] != LABEL].copy()
    for c in BASE_FEATURES + COUNT_COLS + EXTRA_COLS:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if TIME in df:
        df[TIME] = pd.to_datetime(df[TIME], dayfirst=True, errors="coerce")
    df["source_file"] = path.name
    return df

t_start = time.time()
df = pd.concat([load(DATA_DIR / f) for f in FILES], ignore_index=True)
print(df.shape, f"({time.time() - t_start:.0f} s)")

Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv - 84 columns | has Dst Port: True


Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv - 80 columns | has Dst Port: True


(8997323, 25) (49 s)


## 3. Keep benign and DDoS flows, and name the tool (as v2)

In [4]:
def tool_of(label):
    s = str(label).lower()
    if "benign" in s: return "Benign"
    if "udp" in s:    return "LOIC-UDP"
    if "hoic" in s:   return "HOIC"
    if "http" in s:   return "LOIC-HTTP"
    return "Other"

df["tool"] = df[LABEL].map(tool_of)
df = df[df["tool"] != "Other"].copy()
df["y"] = (df["tool"] != "Benign").astype(int)     # 1 = DDoS (positive class)
print(df["tool"].value_counts())

tool
Benign       7733390
HOIC          686012
LOIC-HTTP     576191
LOIC-UDP        1730
Name: count, dtype: int64


## 4. Feature 7: flows to the same destination port in the previous 10 seconds (new)

For a flow that starts in second *s*, feature 7 counts the flows **of the same file** to the **same
destination port** that started in seconds *s*−10 to *s*−1. It looks back only (it could be counted while
traffic arrives) and does not count flows in the same second.

It is counted on **every row of each file, before cleaning**, exactly as the desktop app counts it for the
file it opens (`DDoSDetector/app/core/features.py`, `_port_window`). Times and ports are used only to
count; they are never model inputs. pandas 3 stores these times in microseconds, so seconds are computed
with `total_seconds()` rather than by dividing the raw integer.

In [5]:
def port_window(d):
    sec = np.floor((d[TIME] - pd.Timestamp("1970-01-01")).dt.total_seconds())
    key = pd.DataFrame({"cap": d["source_file"], "port": pd.to_numeric(d[PORT], errors="coerce"),
                        "sec": sec}).dropna()
    c = key.groupby(["cap", "port", "sec"]).size().rename("n").reset_index().sort_values(["cap", "port", "sec"])
    c["before"] = c.groupby(["cap", "port"])["n"].cumsum() - c["n"]             # flows with sec < s
    look = c[["cap", "port", "sec"]].assign(target=c["sec"] - WINDOW_SECONDS).sort_values("target")
    right = c[["cap", "port", "sec", "before", "n"]].rename(columns={"sec": "rsec"}).sort_values("rsec")
    m = pd.merge_asof(look, right, left_on="target", right_on="rsec", by=["cap", "port"],
                      direction="backward", allow_exact_matches=False)
    m["before_window"] = (m["before"] + m["n"]).fillna(0)                        # flows with sec < s-10
    c = c.merge(m[["cap", "port", "sec", "before_window"]], on=["cap", "port", "sec"], how="left")
    c["count"] = c["before"] - c["before_window"]
    rows = key.reset_index().merge(c[["cap", "port", "sec", "count"]], on=["cap", "port", "sec"], how="left")
    out = pd.Series(np.nan, index=d.index)
    out.loc[rows["index"].to_numpy()] = rows["count"].to_numpy(dtype="float64")
    return out

df["dst_port_flows_10s"] = port_window(df)                                       # feature 7

# Check against a brute-force count on 100 random rows
sec = np.floor((df[TIME] - pd.Timestamp("1970-01-01")).dt.total_seconds())
port = pd.to_numeric(df[PORT], errors="coerce")
rng = np.random.default_rng(0)
mismatches = 0
for i in rng.choice(len(df), 100, replace=False):
    same = (df["source_file"].to_numpy() == df["source_file"].iat[i]) & (port.to_numpy() == port.iat[i])
    want = int((same & (sec.to_numpy() >= sec.iat[i] - WINDOW_SECONDS) & (sec.to_numpy() <= sec.iat[i] - 1)).sum())
    mismatches += want != df["dst_port_flows_10s"].iat[i]
print("brute-force mismatches in 100 random rows:", mismatches)
print("rows without a value (no readable time or port):", int(df["dst_port_flows_10s"].isna().sum()))

brute-force mismatches in 100 random rows: 0
rows without a value (no readable time or port): 0


## 5. Clean (as v2; the duplicate check also ignores the port and feature 7, so the row count is unchanged)

In [6]:
n0 = len(df)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=BASE_FEATURES + COUNT_COLS)
# duplicates are judged on the original columns only, so row counts match earlier runs
df = df.drop_duplicates(subset=[c for c in df.columns
                                if c not in EXTRA_COLS + ["source_file", PORT, "dst_port_flows_10s"]])
print(f"rows: {n0:,} -> {len(df):,}")
print(df["tool"].value_counts())

rows: 8,997,323 -> 8,788,935
tool
Benign       7542570
HOIC          668461
LOIC-HTTP     576174
LOIC-UDP        1730
Name: count, dtype: int64


## 6. Build the features (as v2) and the feature sets compared here

In [7]:
df["fwd_bwd_ratio"] = df["Tot Fwd Pkts"] / (df["Tot Bwd Pkts"] + 1)                     # feature 3
df["no_reply"] = (df["Tot Bwd Pkts"] == 0).astype(int)                                  # feature 10
df["empty_payload"] = (df["Fwd Pkt Len Mean"] == 0).astype(int)                         # feature 10
df["iat_cv"] = (df["Flow IAT Std"] / (df["Flow IAT Mean"] + 1)).replace([np.inf, -np.inf], np.nan)  # feature 11
df["bwd_fwd_bytes_ratio"] = df["TotLen Bwd Pkts"] / (df["TotLen Fwd Pkts"] + 1)        # feature 12
df["data_share"] = (df["Fwd Act Data Pkts"] / df["Tot Fwd Pkts"]).replace([np.inf, -np.inf], np.nan)  # feature 15

GROUPS = {   # Week 3 numbering
    1: ("rate", ["Flow Pkts/s", "Flow Byts/s"]), 2: ("duration", ["Flow Duration"]),
    3: ("fwd/bwd ratio", ["fwd_bwd_ratio"]), 4: ("inter-arrival time", ["Flow IAT Mean", "Flow IAT Std"]),
    5: ("SYN flag count", ["SYN Flag Cnt"]), 6: ("forward packet length", ["Fwd Pkt Len Mean", "Fwd Pkt Len Std"]),
    7: ("port window", ["dst_port_flows_10s"]), 8: ("reply behaviour", ["Bwd Pkt Len Mean"]),
    9: ("connection-ending flags", ["RST Flag Cnt", "FIN Flag Cnt"]), 10: ("indicators", ["no_reply", "empty_payload"]),
    11: ("timing regularity", ["iat_cv"]), 12: ("response size ratio", ["bwd_fwd_bytes_ratio"]),
    13: ("sender timing", ["Fwd IAT Mean", "Fwd IAT Min"]), 14: ("activity pattern", ["Active Mean", "Idle Mean"]),
    15: ("payload signals", ["PSH Flag Cnt", "data_share"]),
}
def cols(groups):
    return [c for g in groups for c in GROUPS[g][1]]

F9 = cols((1, 2, 4, 5, 6, 3))         # the original nine columns (v2's FEATURES_BASE9, same order)
NO_SYN = cols((1, 2, 4, 6, 3))        # the same without SYN Flag Cnt (eight columns)
SETS = {"f9 (with SYN)": F9, "no SYN (8 columns)": NO_SYN}
for g in range(7, 16):
    SETS[f"SYN -> F{g} {GROUPS[g][0]}"] = NO_SYN + GROUPS[g][1]
F9P = SETS["SYN -> F7 port window"]   # the desktop app's default set f9p (same columns, same order)
print({name: len(c) for name, c in SETS.items()})
print("f9p:", F9P)

{'f9 (with SYN)': 9, 'no SYN (8 columns)': 8, 'SYN -> F7 port window': 9, 'SYN -> F8 reply behaviour': 9, 'SYN -> F9 connection-ending flags': 10, 'SYN -> F10 indicators': 10, 'SYN -> F11 timing regularity': 9, 'SYN -> F12 response size ratio': 9, 'SYN -> F13 sender timing': 10, 'SYN -> F14 activity pattern': 10, 'SYN -> F15 payload signals': 10}
f9p: ['Flow Pkts/s', 'Flow Byts/s', 'Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'fwd_bwd_ratio', 'dst_port_flows_10s']


## 7. A first look at feature 7

Destination ports and feature 7 per tool. Both HTTP attacks and LOIC-UDP target port 80; most benign
flows go elsewhere.

In [8]:
for t, g in df.groupby("tool"):
    top = g[PORT].value_counts(normalize=True).head(3).round(3).to_dict()
    q = g["dst_port_flows_10s"].quantile([0.1, 0.5, 0.9])
    print(f"{t:<10} {len(g):>9,} flows | top ports {str(top):<46} | feature 7 p10/p50/p90: "
          f"{q.iloc[0]:,.0f} / {q.iloc[1]:,.0f} / {q.iloc[2]:,.0f}")

Benign     7,542,570 flows | top ports {'53': 0.325, '443': 0.161, '3389': 0.145}     | feature 7 p10/p50/p90: 0 / 374 / 1,016
HOIC         668,461 flows | top ports {'80': 1.0}                                    | feature 7 p10/p50/p90: 3,646 / 4,007 / 7,995
LOIC-HTTP    576,174 flows | top ports {'80': 1.0}                                    | feature 7 p10/p50/p90: 1,663 / 1,752 / 2,054
LOIC-UDP       1,730 flows | top ports {'80': 1.0}                                    | feature 7 p10/p50/p90: 10 / 22 / 30


### A quirk of the 21 Feb file

Almost all benign flows in the 21 Feb file fall inside the HOIC attack, about one per HOIC flow, and are
sent to high TCP client ports (in the 50,000s; each port is reused by many flows). They look like the web
server's side of the HOIC connections labelled benign (our reading; the dataset does not say so). Their
feature 7 count is low because no single client port is busy. The "attack period" benign split therefore
mostly tests those flows, not normal users of a busy server. Keep this in mind when reading false alarms
below.

In [9]:
wed = df[df["source_file"] == FILES[1]]
per_min = pd.crosstab(wed[TIME].dt.floor("min"), wed["tool"])
print(per_min[(per_min.index >= "2018-02-21 02:08") & (per_min.index <= "2018-02-21 02:35")].to_string())
in_hoic = wed[TIME].between(wed.loc[wed["tool"] == "HOIC", TIME].min(), wed.loc[wed["tool"] == "HOIC", TIME].max())
b = wed[in_hoic & (wed["tool"] == "Benign")]
print(f"\nbenign flows of the 21 Feb file inside the HOIC attack: {len(b):,} of {int((wed['tool'] == 'Benign').sum()):,}")
print("share of those flows whose destination port appears only once:",
      round(float((b[PORT].map(b[PORT].value_counts()) == 1).mean()), 3))
print("their median destination port:", int(pd.to_numeric(b[PORT]).median()),
      "| median feature 7:", float(b["dst_port_flows_10s"].median()))

tool                 Benign   HOIC  LOIC-UDP
Timestamp                                   
2018-02-21 02:08:00       3      0         0
2018-02-21 02:09:00       1      0         0
2018-02-21 02:10:00       7      0         0
2018-02-21 02:11:00       3  37151         0
2018-02-21 02:12:00       2  47474         0
2018-02-21 02:13:00       3  47326         0
2018-02-21 02:14:00       4  47576         0
2018-02-21 02:15:00       3  47522         0
2018-02-21 02:16:00       9  47547         0
2018-02-21 02:17:00     163  47232         0
2018-02-21 02:18:00   22951  24403         0
2018-02-21 02:19:00   23922  23474         0
2018-02-21 02:20:00   23907  23375         0
2018-02-21 02:21:00   23924  23408         0
2018-02-21 02:22:00   23905  23410         0
2018-02-21 02:23:00   23917  23410         0
2018-02-21 02:24:00   23922  23356         0
2018-02-21 02:25:00   23949  23312         0
2018-02-21 02:26:00   23924  23175         0
2018-02-21 02:27:00   23848  23059         0
2018-02-21

their median destination port: 56825 | median feature 7: 1.0


## 8. Splits (the same as the desktop app's training script)

- **all tools** (random): benign capped at 1,000,000 plus every attack flow, random 70/30 split stratified
  by tool. Optimistic: near-identical flows from the same burst land in both halves, and for feature 7 the
  training and test flows share the same seconds.
- **held out** (one tool never seen in training), benign split:
  `per_day` = the earlier 70% of each file's benign flows train, the later 30% test (v2 section 7);
  `by_file` = the earlier file's benign flows train, the later file's test (v2 section 7);
  `attack_period` = benign flows inside the held-out tool's attack window test, all others train.

In [10]:
def split(setup, benign_mode, seed):
    if setup == "all tools":
        ben = df[df["tool"] == "Benign"]
        d = pd.concat([ben.sample(min(BENIGN_CAP, len(ben)), random_state=seed), df[df["tool"] != "Benign"]])
        return train_test_split(d, test_size=1 - TRAIN_FRAC, stratify=d["tool"], random_state=seed)
    d = df[df[TIME].notna()]
    ben, att = d[d["tool"] == "Benign"], d[d["tool"] != "Benign"]
    held = att[att["tool"] == setup]
    if benign_mode == "per_day":
        in_test = ben.groupby("source_file")[TIME].rank(pct=True, method="first") > TRAIN_FRAC
    elif benign_mode == "by_file":
        order = ben.groupby("source_file")[TIME].min().sort_values().index
        in_test = ben["source_file"] == order[-1]
    else:  # attack_period
        in_test = (ben["source_file"].isin(held["source_file"].unique())
                   & ben[TIME].between(held[TIME].min(), held[TIME].max()))
    tr = pd.concat([ben[~in_test].sample(min(int((~in_test).sum()), BENIGN_CAP), random_state=seed),
                    att[att["tool"] != setup]])
    te = pd.concat([ben[in_test], held])
    return tr, te

## 9. Models

- **XGBoost**: the project's model (boosted trees), settings as in the app.
- **Random Forest**: 100 trees, maximum depth 20, at least 2 flows per leaf, `class_weight="balanced"`
  (the counterpart of XGBoost's `scale_pos_weight`). Bagged trees: each tree is trained on a random
  sample and the trees vote.
- **Logistic regression**: a weighted sum of the features. Rates and counts span many orders of
  magnitude, so the columns are log-scaled (keeping the sign) and standardised first;
  `class_weight="balanced"`.

None of the settings was tuned.

In [11]:
def xgboost(spw, seed):
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist",
                         scale_pos_weight=spw, n_jobs=-1, random_state=seed)

def random_forest(spw, seed):
    return RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2,
                                  class_weight="balanced", n_jobs=-1, random_state=seed)

def signed_log(X):
    X = np.asarray(X, dtype=float)
    return np.sign(X) * np.log1p(np.abs(X))

def logistic(spw, seed):
    return make_pipeline(FunctionTransformer(signed_log), StandardScaler(),
                         LogisticRegression(class_weight="balanced", max_iter=2000))

MODELS = {"XGBoost": xgboost, "Random Forest": random_forest, "Logistic regression": logistic}

def evaluate(model, te, features):
    p = model.predict_proba(te[features])[:, 1]
    y, tool, pred = te["y"].to_numpy(), te["tool"].to_numpy(), p >= 0.5
    out = {"fpr": float(pred[y == 0].mean()), "false_alarms": int(pred[y == 0].sum()),
           "pr_auc": float(average_precision_score(y, p))}
    for t in ["HOIC", "LOIC-HTTP", "LOIC-UDP"]:
        if (tool == t).any():
            out[f"recall {t}"] = float(pred[tool == t].mean())
    return out

## 10. Run every experiment (about an hour)

- **A. SYN swap (XGBoost):** the nine columns, the eight without SYN, and SYN replaced by each of features
  7 to 15, on every split below, two seeds.
- **B. Model comparison:** Random Forest and logistic regression on the nine columns (f9) and on f9p (SYN
  replaced by feature 7), on four of the splits, two seeds. XGBoost's numbers for B come from A (same
  splits, same seeds).

In [12]:
SETUPS = [("all tools", "random"), ("HOIC", "per_day"), ("HOIC", "by_file"), ("HOIC", "attack_period"),
          ("LOIC-HTTP", "per_day"), ("LOIC-HTTP", "attack_period"),
          ("LOIC-UDP", "per_day"), ("LOIC-UDP", "attack_period")]
COMPARE_SETUPS = [("all tools", "random"), ("HOIC", "per_day"), ("HOIC", "attack_period"), ("LOIC-HTTP", "per_day")]
COMPARE_SETS = ["f9 (with SYN)", "SYN -> F7 port window"]

rows, t0 = [], time.time()
for setup, mode in SETUPS:
    for seed in SEEDS:
        tr, te = split(setup, mode, seed)
        spw = (tr["y"] == 0).sum() / (tr["y"] == 1).sum()
        runs = [("XGBoost", name) for name in SETS]
        if (setup, mode) in COMPARE_SETUPS:
            runs += [(m, name) for m in ("Random Forest", "Logistic regression") for name in COMPARE_SETS]
        for model_name, set_name in runs:
            features = SETS[set_name]
            t1 = time.perf_counter()
            model = MODELS[model_name](spw, seed).fit(tr[features], tr["y"])
            rows.append({"setup": setup if setup == "all tools" else f"{setup} held out", "benign split": mode,
                         "seed": seed, "model": model_name, "feature set": set_name, "columns": len(features),
                         "train s": round(time.perf_counter() - t1, 1), **evaluate(model, te, features)})
    print(f"{setup} / {mode} done ({time.time() - t0:.0f} s)")
results = pd.DataFrame(rows)
print(len(results), "models trained")

all tools / random done (403 s)


HOIC / per_day done (782 s)


HOIC / by_file done (965 s)


HOIC / attack_period done (1303 s)


LOIC-HTTP / per_day done (1707 s)


LOIC-HTTP / attack_period done (2000 s)


LOIC-UDP / per_day done (2304 s)


LOIC-UDP / attack_period done (2567 s)
208 models trained


## 11. Check: the f9p XGBoost results match the desktop app's trained models

The app's models (`DDoSDetector/models/*_f9p.meta.json`, seed 0) were trained by `train_model.py` on the
same data, splits and settings, so seed 0 here should give the same recall and false positive rate.

In [13]:
import json
MODELS_DIR = Path("../DDoSDetector/models")
NAMES = {("all tools", "random"): "all_tools_f9p", ("HOIC held out", "per_day"): "hoic_held_out_f9p",
         ("HOIC held out", "by_file"): "hoic_held_out_by_file_f9p",
         ("HOIC held out", "attack_period"): "hoic_held_out_attack_period_f9p",
         ("LOIC-HTTP held out", "per_day"): "loic_http_held_out_f9p",
         ("LOIC-HTTP held out", "attack_period"): "loic_http_held_out_attack_period_f9p",
         ("LOIC-UDP held out", "per_day"): "loic_udp_held_out_f9p",
         ("LOIC-UDP held out", "attack_period"): "loic_udp_held_out_attack_period_f9p"}
check = []
for (setup, mode), name in NAMES.items():
    p = MODELS_DIR / f"{name}.meta.json"
    if not p.exists():
        continue
    meta = json.loads(p.read_text(encoding="utf-8"))["metrics"]
    r = results[(results["setup"] == setup) & (results["benign split"] == mode) & (results["seed"] == 0)
                & (results["model"] == "XGBoost") & (results["feature set"] == "SYN -> F7 port window")].iloc[0]
    for t, rec in meta["recall"].items():
        check.append({"model": name, "tool": t, "recall (app)": rec, "recall (here)": round(r[f"recall {t}"], 4),
                      "FPR (app)": meta["fpr"], "FPR (here)": round(r["fpr"], 4)})
print(pd.DataFrame(check).to_string(index=False) if check else "no f9p models found in ../DDoSDetector/models")

                               model      tool  recall (app)  recall (here)  FPR (app)  FPR (here)
                       all_tools_f9p      HOIC        1.0000         1.0000     0.0003      0.0003
                       all_tools_f9p LOIC-HTTP        0.9998         0.9998     0.0003      0.0003
                       all_tools_f9p  LOIC-UDP        1.0000         1.0000     0.0003      0.0003
                   hoic_held_out_f9p      HOIC        0.3967         0.3967     0.0020      0.0020
           hoic_held_out_by_file_f9p      HOIC        0.3694         0.3694     0.0000      0.0000
     hoic_held_out_attack_period_f9p      HOIC        0.3621         0.3621     0.0000      0.0000
              loic_http_held_out_f9p LOIC-HTTP        0.0023         0.0023     0.0005      0.0005
loic_http_held_out_attack_period_f9p LOIC-HTTP        0.0025         0.0025     0.0045      0.0045
               loic_udp_held_out_f9p  LOIC-UDP        0.9936         0.9936     0.0023      0.0023
 loic_udp_

## 12. A. Replacing SYN flag count (XGBoost)

Ranges are over the two seeds. "vs f9" applies the group's rule: **better** only if the lowest recall over
the seeds is above the highest recall of the nine columns and the highest false positive rate is not above
theirs; **worse** if the highest recall is below the nine columns' lowest. For "all tools" the recall column
is the lowest per-tool recall.

In [14]:
def held_recall(r):
    if r["setup"] == "all tools":
        return min(r["recall HOIC"], r["recall LOIC-HTTP"], r["recall LOIC-UDP"])
    return r[f"recall {r['setup'].replace(' held out', '')}"]

results["recall"] = results.apply(held_recall, axis=1)

def span(s, fmt="{:.3f}"):
    lo, hi = s.min(), s.max()
    return fmt.format(lo) if fmt.format(lo) == fmt.format(hi) else f"{fmt.format(lo)}–{fmt.format(hi)}"

def table(sub, base_name):
    base = sub[sub["feature set"] == base_name]
    out = []
    for name, g in sub.groupby("feature set", sort=False):
        better = g["recall"].min() > base["recall"].max() and g["fpr"].max() <= base["fpr"].max()
        worse = g["recall"].max() < base["recall"].min()
        out.append({"feature set": name, "recall": span(g["recall"]), "FPR": span(g["fpr"], "{:.4f}"),
                    "false alarms": span(g["false_alarms"], "{:,.0f}"), "PR-AUC": span(g["pr_auc"]),
                    "vs f9": "better" if better else "worse" if worse else ""})
    return pd.DataFrame(out)

A = results[results["model"] == "XGBoost"]
for (setup, mode), sub in A.groupby(["setup", "benign split"], sort=False):
    print(f"\n=== {setup} / {mode}")
    print(table(sub, "f9 (with SYN)").to_string(index=False))


=== all tools / random
                      feature set      recall           FPR false alarms PR-AUC  vs f9
                    f9 (with SYN) 0.989–0.990 0.0104–0.0108  3,133–3,243  0.999       
               no SYN (8 columns) 0.989–0.990 0.0104–0.0108  3,133–3,243  0.999       
            SYN -> F7 port window       1.000 0.0002–0.0003       70–103  1.000 better
        SYN -> F8 reply behaviour 0.987–0.988 0.0091–0.0097  2,739–2,896  0.999  worse
SYN -> F9 connection-ending flags 0.987–0.988 0.0091–0.0097  2,730–2,907  0.999  worse
            SYN -> F10 indicators 0.987–0.990 0.0091–0.0108  2,738–3,236  0.999       
     SYN -> F11 timing regularity 0.987–0.988 0.0092–0.0097  2,747–2,911  0.999  worse
   SYN -> F12 response size ratio 0.987–0.988 0.0092–0.0097  2,747–2,918  0.999  worse
         SYN -> F13 sender timing       0.988 0.0096–0.0097  2,872–2,919  0.999  worse
      SYN -> F14 activity pattern 0.988–0.989 0.0081–0.0086  2,426–2,579  0.999       
       SYN -> F15 p

## 13. B. XGBoost vs Random Forest vs logistic regression

Same splits and seeds for all three models. `f9` = the original nine columns; `SYN -> F7 port window` = f9p,
the app's default since update 4.

In [15]:
B = results[results["feature set"].isin(COMPARE_SETS)
            & results.set_index(["setup", "benign split"]).index.isin(
                [(s if s == "all tools" else f"{s} held out", m) for s, m in COMPARE_SETUPS])]
rows_b = []
for (setup, mode, fs, model), g in B.groupby(["setup", "benign split", "feature set", "model"], sort=False):
    row = {"setup": setup, "benign split": mode, "feature set": fs, "model": model}
    for t in ["HOIC", "LOIC-HTTP", "LOIC-UDP"]:
        if f"recall {t}" in g and g[f"recall {t}"].notna().any():
            row[f"recall {t}"] = span(g[f"recall {t}"])
    row.update({"FPR": span(g["fpr"], "{:.4f}"), "false alarms": span(g["false_alarms"], "{:,.0f}"),
                "PR-AUC": span(g["pr_auc"]), "train s": f"{g['train s'].mean():.0f}"})
    rows_b.append(row)
print(pd.DataFrame(rows_b).fillna("").to_string(index=False))

             setup  benign split           feature set               model recall HOIC recall LOIC-HTTP recall LOIC-UDP           FPR    false alarms      PR-AUC train s
         all tools        random         f9 (with SYN)             XGBoost       0.999      0.989–0.990           1.000 0.0104–0.0108     3,133–3,243       0.999       8
         all tools        random SYN -> F7 port window             XGBoost       1.000            1.000           1.000 0.0002–0.0003          70–103       1.000       8
         all tools        random         f9 (with SYN)       Random Forest       0.999            0.996           0.998 0.0085–0.0087     2,558–2,607       0.999      44
         all tools        random SYN -> F7 port window       Random Forest       1.000            1.000           1.000        0.0002           47–57       1.000      46
         all tools        random         f9 (with SYN) Logistic regression 0.853–0.855            0.988           0.000 0.0749–0.0756   22,472–22,671 

## 14. Findings

31 Oct 2026, from the tables above. Two seeds, untuned settings, threshold 0.5.

**Checks**
- Feature 7 matches a brute-force count (0 mismatches in 100 random rows) and every row has a value.
  Cleaning still gives 8,788,935 rows.
- The XGBoost f9p results here equal the desktop app's trained models to four decimals (section 11), so
  the notebook and the app compute the same thing.

**A. Replacing SYN flag count (XGBoost)**
- Removing SYN changes nothing: "no SYN" gives exactly the nine-column numbers in every split.
- **Feature 7 is the only replacement that passes the group's rule on all three HOIC held-out splits**:
  unseen HOIC recall 0.395–0.397 (per day), 0.364–0.369 (by file) and 0.362–0.363 (attack period), against
  0.165–0.212, 0.145–0.149 and 0.147–0.153 with the nine columns, with fewer or the same false alarms
  (per day 4,282–4,514 against 17,493–17,764).
- Features 8 to 15 pass only on the by-file split, where one seed reaches 0.199 for almost all of them.
  That is a seed effect, not something one feature does; none passes on the per-day or attack-period
  split. Feature 14 (activity pattern) lowers false alarms by about a quarter in most splits without
  changing recall.
- Feature 7 does **not** help unseen LOIC-HTTP (0.002–0.003). It ranks LOIC-HTTP flows much higher
  (PR-AUC 0.91–0.94 against about 0.41), but most stay below 0.5. LOIC-HTTP sends about 1,750 flows per
  10 s to port 80, HOIC about 4,000, so a model that learned HOIC's volume does not fire at LOIC-HTTP's.
  On LOIC-UDP it misses 11 of 1,730 flows (0.994) that the nine columns caught. It lowers false alarms in
  every held-out split.

**B. XGBoost, Random Forest and logistic regression**
- Nine columns: XGBoost and Random Forest are close (unseen HOIC per day 0.165–0.212 and 0.166–0.172);
  logistic regression catches no unseen HOIC and has many more false alarms. Random Forest trains about
  five times slower than XGBoost.
- f9p: Random Forest gives about the same as XGBoost (unseen HOIC 0.393, LOIC-HTTP 0.002–0.003).
  **Logistic regression changes the most**: unseen HOIC 0.859–0.861 and unseen LOIC-HTTP 0.487 (per day),
  but with more false alarms: 27,315–29,412 and 56,206–56,908 benign flows (false positive rates
  0.012–0.013 and 0.025), against 4,282–4,514 and 966–1,218 for XGBoost.
- Our reading, not tested: logistic regression treats "more flows to the port" as steadily more
  attack-like, so it carries over to volumes it has not seen, while trees learn cut-offs at the volumes in
  their training data. A lower threshold on the XGBoost model might reach a similar trade-off (its PR-AUC
  on unseen LOIC-HTTP is 0.91–0.94); that has not been tried.

**Caveats**
- The all-tools (random split) numbers with feature 7 are not trustworthy: training and test flows share
  the same seconds of the same attack.
- In the 21 Feb file, 358,619 of 360,808 benign flows fall inside the HOIC attack (see section 7). Zero
  false alarms in the attack-period split therefore say little about normal users of a busy server.
- Feature 7 counts by destination port, so it mixes every server using that port, and its values depend on
  how busy the network is. Normal flows to a busy port during an attack or a flash crowd get high counts
  too; this dataset has no labelled flash crowd to test that.
- LOIC-UDP has only 1,730 flows. Two seeds only; no setting was tuned.